# A1 — Redeveloped LLM (Layer 3)
**COMP8420 2026 S1 — BEACON Brand Monitoring**

Multi-task RoBERTa: crisis severity, sentiment, and topic from a single Reddit post.

## 1. Objective

**Technique:** A redeveloped multi-task foundation model with a shared **RoBERTa-base encoder** and three task-specific heads, trained jointly with weighted losses while **fine-tuning only the top encoder layers** and freezing the bottom stack. It predicts:

- **Crisis severity** (0–3): 0 = no concern, 1 = minor complaint, 2 = escalating concern, 3 = active crisis
- **Fine-grained sentiment** (−1.0 to 1.0, regression head)
- **Topic** (fixed 8-category brand-monitoring taxonomy)

**Why this matters for social media brand monitoring:** One shared encoder pass per post yields three specialised predictions, avoiding three separate model calls at inference time. Joint training lets related signals (e.g., sentiment and crisis severity) reinforce each other.

*Note:* A Q-Former-style alternative was attempted first but abandoned due to data-size constraints; the full evidence and rationale are documented in Sections 6–7.

## 2. Setup

Imports, reproducibility seeds, data loading from `load_sample()`, LLM pseudo-labeling, the manually reviewed eval holdout, and the train/val split.

In [ ]:
# Colab bootstrap (run once at the top of the notebook)
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call(["/bin/bash", "-lc", "rm -rf beacon"])
    subprocess.check_call(["/bin/bash", "-lc", "git clone https://github.com/vutuongvy101/beacon.git"])
    os.chdir("/content/beacon")
    subprocess.check_call(["/bin/bash", "-lc", "git log -1 --oneline"])
    subprocess.check_call(["/bin/bash", "-lc", "ls advanced/layer3_llm/"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "datasets", "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn",
        "tqdm", "bertopic", "sentence-transformers", "umap-learn", "emoji", "contractions", "joblib",
    ])
    print("Colab setup complete.")
else:
    print("Not running on Colab; skip this cell.")

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

from datasets import Dataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from openai import OpenAI
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer

ROOT = Path.cwd().resolve()
candidates = [ROOT]
if ROOT.name == "advanced":
    candidates.append(ROOT.parent)
candidates.append(Path.cwd().resolve().parent)
for candidate in candidates:
    if (candidate / "shared").exists() and (candidate / "advanced" / "layer3_llm").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Cannot find beacon repo root. Run this notebook from the repo root.")
sys.path.insert(0, str(ROOT))

from shared.data_loader import load_sample
from shared.preprocessing import clean_for_llm
from advanced.layer3_llm.baseline_compare import (
    evaluate_pipeline_baseline,
    pipeline_trainable_param_count,
)
from advanced.layer3_llm.multitask_models import (
    LOSS_WEIGHTS,
    StandardMultiTaskRoberta,
    TOPIC_LABELS,
)
from advanced.layer3_llm.train_utils import (
    MultiTaskDataset,
    evaluate_predictions,
    get_device,
    load_checkpoint,
    measure_inference_latency_ms,
    predict_batch,
    save_checkpoint,
    train_model,
)

LAYER3_DIR = ROOT / "advanced" / "layer3_llm"
OUTPUT_DIR = LAYER3_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PSEUDO_CSV = OUTPUT_DIR / "pseudo_labels.csv"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = get_device()
print(f"Project root: {ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")
print(f"Device:       {DEVICE}  (set LAYER3_DEVICE=cpu|cuda to override)")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
print("OpenAI API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
df = load_sample(brand="openai", as_df=True)
TEXT_COL = "text_for_llm"

if "title" in df.columns or "selftext" in df.columns:
    title = df["title"] if "title" in df.columns else pd.Series("", index=df.index)
    body = df["selftext"] if "selftext" in df.columns else pd.Series("", index=df.index)
    raw_text = title.fillna("").astype(str) + "\n" + body.fillna("").astype(str)
elif "text" in df.columns:
    raw_text = df["text"].fillna("").astype(str)
elif "body" in df.columns:
    raw_text = df["body"].fillna("").astype(str)
else:
    raw_text = df.iloc[:, 0].fillna("").astype(str)

df[TEXT_COL] = clean_for_llm(raw_text.tolist())
df = df[df[TEXT_COL].astype(str).str.len() > 20].reset_index(drop=True)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.slice(0, 1500)

display(df[["post_id", TEXT_COL]].head(3))
print(f"\nCorpus: {len(df)} posts")

### Pseudo-labeling rubric (LLM)

We label each post with:
- **Crisis severity** (0–3): 0 = no concern, 1 = minor complaint, 2 = escalating concern, 3 = active crisis
- **Sentiment score** in **[-1.0, 1.0]**
- **Topic** from the fixed taxonomy

The prompt below includes 2 examples per severity level to keep the rubric consistent.

In [ ]:
LABEL_MODEL = "gpt-4o-mini"
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

FEW_SHOT = [
    {"text": "OpenAI is amazing, the new model is faster and cheaper.", "crisis_severity": 0, "sentiment_score": 0.8, "topic": "product_releases", "rationale": "positive update"},
    {"text": "Nothing wrong, just sharing a cool demo of the API.", "crisis_severity": 0, "sentiment_score": 0.4, "topic": "api_developer", "rationale": "neutral/positive"},
    {"text": "Subscription price went up again and I'm annoyed.", "crisis_severity": 1, "sentiment_score": -0.4, "topic": "pricing_subscriptions", "rationale": "minor complaint"},
    {"text": "Support is slow and billing keeps failing for me.", "crisis_severity": 1, "sentiment_score": -0.6, "topic": "pricing_subscriptions", "rationale": "frustration but not crisis"},
    {"text": "The API outage has lasted hours; our deployment is blocked.", "crisis_severity": 2, "sentiment_score": -0.7, "topic": "reliability_outages", "rationale": "escalating operational impact"},
    {"text": "Customer trust is slipping after the latest safety incident.", "crisis_severity": 2, "sentiment_score": -0.6, "topic": "safety_ethics", "rationale": "escalating concern"},
    {"text": "Major breach reported and services are down globally.", "crisis_severity": 3, "sentiment_score": -0.9, "topic": "reliability_outages", "rationale": "active crisis"},
    {"text": "Breaking news: regulators are investigating OpenAI leadership for misconduct.", "crisis_severity": 3, "sentiment_score": -0.8, "topic": "corporate_leadership", "rationale": "active crisis"},
]

TOPIC_LIST = ", ".join(TOPIC_LABELS)
SYSTEM_PROMPT = (
    "You are labeling Reddit posts for brand monitoring. "
    "Return STRICT JSON with keys: post_id, text, crisis_severity (0-3), sentiment_score (-1 to 1), "
    "topic (one of: " + TOPIC_LIST + "), rationale."
)

def build_prompt(batch_records: list[dict]) -> str:
    few_shot = "\n".join([json.dumps(x, ensure_ascii=False) for x in FEW_SHOT])
    payload = json.dumps(batch_records, ensure_ascii=False)
    return (
        "Examples:\n" + few_shot + "\n\n"
        "Label these posts as a JSON array with the same keys:\n" + payload
    )


In [ ]:
def label_batch(batch_df: pd.DataFrame) -> list[dict]:
    records = batch_df[["post_id", TEXT_COL]].rename(columns={TEXT_COL: "text"}).to_dict("records")
    text_map = {str(r["post_id"]): r["text"] for r in records}
    prompt = build_prompt(records)
    response = client.chat.completions.create(
        model=LABEL_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )
    content = response.choices[0].message.content or ""
    try:
        parsed = json.loads(content)
        if isinstance(parsed, list):
            for item in parsed:
                pid = str(item.get("post_id", ""))
                if "text" not in item:
                    item["text"] = text_map.get(pid, "")
            return parsed
        return []
    except json.JSONDecodeError:
        return []

def normalize_label(row: dict) -> dict | None:
    try:
        return {
            "post_id": str(row["post_id"]),
            "text": str(row.get("text", ""))[:1500],
            "crisis_severity": int(row["crisis_severity"]),
            "sentiment_score": float(row["sentiment_score"]),
            "topic": str(row["topic"]),
            "rationale": str(row.get("rationale", ""))[:300],
        }
    except Exception:
        return None

In [ ]:
if PSEUDO_CSV.exists():
    pseudo_df = pd.read_csv(PSEUDO_CSV)
    print(f"Loaded {PSEUDO_CSV} ({len(pseudo_df)} rows)")
else:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is not set. Export it before running labeling.")
    batch_size = 25
    records: list[dict] = []
    failures: list[str] = []
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i + batch_size]
        labels = label_batch(batch)
        if not labels:
            failures.extend(batch["post_id"].astype(str).tolist())
            continue
        for item in labels:
            normalized = normalize_label(item)
            if normalized is None:
                failures.append(str(item.get("post_id", "unknown")))
                continue
            records.append(normalized)
    pseudo_df = pd.DataFrame(records)
    pseudo_df.to_csv(PSEUDO_CSV, index=False)
    print(f"Saved {PSEUDO_CSV} ({len(pseudo_df)} rows)")
    if failures:
        print(f"Failed labels: {len(failures)} posts")
        print("Sample failed post_ids:", failures[:5])

print("\nSeverity distribution:")
display(pseudo_df["crisis_severity"].value_counts().sort_index())
print("\nTopic distribution:")
display(pseudo_df["topic"].value_counts())
display(pseudo_df[["post_id", "crisis_severity", "sentiment_score", "topic"]].head(5))

In [ ]:
EVAL_TO_REVIEW = OUTPUT_DIR / "eval_set_to_review.csv"
TARGET_EVAL = 300
class_weights = {0: 0.2, 1: 0.25, 2: 0.25, 3: 0.3}

eval_samples = []
for sev, weight in class_weights.items():
    pool = pseudo_df[pseudo_df["crisis_severity"] == sev]
    n = max(1, int(TARGET_EVAL * weight))
    if len(pool) == 0:
        continue
    eval_samples.append(pool.sample(n=n, replace=len(pool) < n, random_state=SEED))

eval_df = pd.concat(eval_samples).drop_duplicates("post_id")
eval_df = eval_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
eval_df.to_csv(EVAL_TO_REVIEW, index=False)
print(f"Saved {EVAL_TO_REVIEW} ({len(eval_df)} rows) — review and correct labels.")

### Manual review required

Open `eval_set_to_review.csv`, correct the labels, and save it as `eval_set_reviewed.csv` in the same `outputs/` folder. The next cell will fail clearly until this reviewed file exists.

In [ ]:
EVAL_GOLD = OUTPUT_DIR / "eval_set_reviewed.csv"
if not EVAL_GOLD.exists():
    raise FileNotFoundError(
        f"Missing {EVAL_GOLD}. Review eval_set_to_review.csv and save the corrected file."
    )
eval_gold = pd.read_csv(EVAL_GOLD)
print(f"Loaded reviewed eval set: {EVAL_GOLD} ({len(eval_gold)} posts)")
print("\nEval severity distribution:")
display(eval_gold["crisis_severity"].value_counts().sort_index())

eval_ids = set(eval_gold["post_id"].astype(str))
train_pool = pseudo_df[~pseudo_df["post_id"].astype(str).isin(eval_ids)].copy()
train_df, val_df = train_test_split(
    train_pool,
    test_size=0.15,
    random_state=SEED,
    stratify=train_pool["crisis_severity"],
)
assert eval_ids.isdisjoint(set(train_df["post_id"].astype(str)))
assert eval_ids.isdisjoint(set(val_df["post_id"].astype(str)))
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Eval holdout: {len(eval_ids)}")

## 3. Implementation

**Standard multi-task RoBERTa (recommended for small data):** shared `[CLS]` representation with three heads. We **partially unfreeze** the top 2 layers of `roberta-base` to adapt to the domain without overfitting.

- `crisis_head`: 4-way softmax (CrossEntropy, class-weighted)
- `sentiment_head`: scalar regression (MSE, clamped to [−1, 1] at inference)
- `topic_head`: 8-way softmax (CrossEntropy)

Combined loss uses task weights tuned to match loss scales. We auto-adjust after epoch 1 so each task contributes comparably (starting from crisis 1.0, sentiment 8.0, topic 0.4).

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 30
LR = 2e-4
MODEL_NAME = "roberta-base"
UNFREEZE_LAST_N = 2
EARLY_STOPPING_PATIENCE = 5
CHECKPOINT_PATH = OUTPUT_DIR / "multitask_model.pt"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
train_ds = MultiTaskDataset(train_df, tokenizer, text_col="text", max_length=MAX_LENGTH)
val_ds = MultiTaskDataset(val_df, tokenizer, text_col="text", max_length=MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

crisis_counts = train_df["crisis_severity"].value_counts().sort_index()
counts = pd.Series([crisis_counts.get(i, 0) for i in range(4)], index=range(4))
crisis_weights = (1.0 / counts.replace(0, np.nan)).fillna(0.0)
if crisis_weights.sum() > 0:
    crisis_weights = crisis_weights / crisis_weights.sum() * len(crisis_weights)
crisis_weights_t = torch.tensor(crisis_weights.values, dtype=torch.float, device=DEVICE)

model = StandardMultiTaskRoberta(
    model_name=MODEL_NAME,
    num_topics=len(TOPIC_LABELS),
    unfreeze_last_n=UNFREEZE_LAST_N,
    crisis_class_weights=crisis_weights_t,
    loss_weights=LOSS_WEIGHTS,
)
print(f"Model trainable parameters: {model.count_trainable_params():,}")

model_history, model_train_time = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=DEVICE,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    auto_loss_weights=True,
)

model_config = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "loss_weights": model.loss_weights,
    "unfreeze_last_n": UNFREEZE_LAST_N,
}
save_checkpoint(
    CHECKPOINT_PATH,
    model,
    arch="standard",
    config=model_config,
    history=model_history,
)
print(f"Checkpoint saved. Train time: {model_train_time:.1f}s")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
metrics = [
    ("train_loss", "val_loss", "Total loss"),
    ("train_loss_crisis", "val_loss_crisis", "Crisis loss"),
    ("train_loss_sentiment", "val_loss_sentiment", "Sentiment loss"),
    ("train_loss_topic", "val_loss_topic", "Topic loss"),
]
for ax, (tk, vk, title) in zip(axes.flat[:4], metrics):
    ax.plot(model_history[tk], label="train")
    ax.plot(model_history[vk], label="val")
    ax.set_title(title)
    ax.legend()
    ax.set_xlabel("epoch")
for ax in axes.flat[4:]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "multitask_loss_curves.png", dpi=120)
plt.show()

## 4. Results

Evaluate the trained multi-task model on the human-reviewed holdout set.

In [ ]:
eval_ds = MultiTaskDataset(eval_gold, tokenizer, text_col="text", max_length=MAX_LENGTH)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)

ckpt = load_checkpoint(CHECKPOINT_PATH, DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.to(DEVICE)

preds = predict_batch(model, eval_loader, DEVICE)
topic_to_id = {t: i for i, t in enumerate(TOPIC_LABELS)}
y_topic_true = [topic_to_id.get(str(t), topic_to_id["general_discussion"]) for t in eval_gold["topic"]]

model_metrics = evaluate_predictions(
    eval_gold["crisis_severity"].tolist(), preds["crisis_pred"],
    eval_gold["sentiment_score"].tolist(), preds["sentiment_pred"],
    y_topic_true, preds["topic_pred"],
)
metrics_df = pd.DataFrame([model_metrics]).drop(columns=["crisis_confusion_matrix"])
display(metrics_df.T)
metrics_df.to_csv(OUTPUT_DIR / "multitask_eval_metrics.csv", index=False)

cm = np.array(model_metrics["crisis_confusion_matrix"])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0,1,2,3], yticklabels=[0,1,2,3])
plt.xlabel("Predicted severity"); plt.ylabel("True severity"); plt.title("Crisis confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "multitask_confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
examples = eval_gold.copy()
examples["pred_crisis"] = preds["crisis_pred"]
examples["pred_sentiment"] = np.round(preds["sentiment_pred"], 3)
examples["pred_topic"] = [TOPIC_LABELS[i] for i in preds["topic_pred"]]
examples["crisis_ok"] = examples["crisis_severity"] == examples["pred_crisis"]
examples["topic_ok"] = examples["topic"] == examples["pred_topic"]
examples["sent_err"] = (examples["sentiment_score"] - examples["pred_sentiment"]).abs()

successes = examples[examples["crisis_ok"] & examples["topic_ok"] & (examples["sent_err"] < 0.3)].head(3)
failures = examples.sort_values("sent_err", ascending=False).head(3)
show = pd.concat([successes, failures]).drop_duplicates("post_id").head(10)
show["text_snip"] = show["text"].str.slice(0, 120) + "..."
display(show[["text_snip", "crisis_severity", "pred_crisis", "sentiment_score", "pred_sentiment", "topic", "pred_topic"]])

## 5. Comparison

Compare the **standard multi-task model** (one forward pass, three heads) against the **existing BEACON pipeline** assembled from basic-layer components:

| Task | Pipeline baseline |
|------|-------------------|
| Crisis severity | **TF-IDF + Logistic Regression** (trained on ChatGPT pseudo-labels) |
| Sentiment | **B4** — `classify_sentiment_detailed()` → `basic/sentiment_model.pkl` + `tfidf_vectorizer.pkl` |
| Topic | **B5** — `fit_bertopic()` + `assign_thread_topics()` from `shared/topics.py` |

No dashboard JSON stubs — we run the same exported functions and model artefacts as B4/B5.

In [ ]:
CRISIS_LR_CKPT = OUTPUT_DIR / "crisis_lr_baseline.pkl"
BERTOPIC_CKPT = OUTPUT_DIR / "bertopic_b5_baseline"

pipeline_metrics, crisis_vec, crisis_clf, bertopic_model, pipeline_train_time = (
    evaluate_pipeline_baseline(
        eval_gold,
        train_pool,
        crisis_ckpt=CRISIS_LR_CKPT,
        bertopic_ckpt=BERTOPIC_CKPT,
        random_state=SEED,
    )
)
print(f"Pipeline fit time (crisis LR + B5 BERTopic): {pipeline_train_time:.2f}s")
print(f"Crisis LR coefficients: {pipeline_trainable_param_count(crisis_vec, crisis_clf):,}")
print(f"B5 topics discovered: {len(bertopic_model.get_topic_info()) - 1}")  # minus outlier row

comparison = pd.DataFrame([
    {
        "model": "Multi-task RoBERTa",
        "crisis_f1": model_metrics["crisis_f1_macro"],
        "sentiment_mae": model_metrics["sentiment_mae"],
        "topic_f1": model_metrics["topic_f1_macro"],
    },
    {
        "model": "B4 sentiment_ml",
        "crisis_f1": np.nan,
        "sentiment_mae": pipeline_metrics["sentiment_mae"],
        "topic_f1": np.nan,
    },
    {
        "model": "B5 topic_clustering",
        "crisis_f1": np.nan,
        "sentiment_mae": np.nan,
        "topic_f1": pipeline_metrics["topic_f1_macro"],
    },
    {
        "model": "TF-IDF + LR crisis",
        "crisis_f1": pipeline_metrics["crisis_f1_macro"],
        "sentiment_mae": np.nan,
        "topic_f1": np.nan,
    },
])
display(comparison)
comparison.to_csv(OUTPUT_DIR / "comparison_table.csv", index=False)

print("\nPer-component note: B4 uses basic/sentiment_model.pkl; B5 fits BERTopic on train pool; crisis LR fit here.")

In [ ]:
labels = ["Crisis F1", "Topic F1", "Sentiment (1-MAE/2)"]
x = np.arange(len(labels))
width = 0.35

mt_scores = [
    model_metrics["crisis_f1_macro"],
    model_metrics["topic_f1_macro"],
    1 - model_metrics["sentiment_mae"] / 2,
]
baseline_scores = [
    pipeline_metrics["crisis_f1_macro"],
    pipeline_metrics["topic_f1_macro"],
    1 - pipeline_metrics["sentiment_mae"] / 2,
]

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.bar(x - width/2, mt_scores, width, label="Multi-task RoBERTa")
ax.bar(x + width/2, baseline_scores, width, label="Basic baselines")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
ax.set_ylim(0, 1); ax.set_title("Per-task scores")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparison_charts.png", dpi=120)
plt.show()

## 6. Justification

Evidence-based comparison for the report (auto-generated from Section 5 results).

In [ ]:
mt_row = comparison[comparison["model"] == "Multi-task RoBERTa"].iloc[0]
b4_row = comparison[comparison["model"] == "B4 sentiment_ml"].iloc[0]
b5_row = comparison[comparison["model"] == "B5 topic_clustering"].iloc[0]
lr_row = comparison[comparison["model"] == "TF-IDF + LR crisis"].iloc[0]

crisis_winner = "Multi-task" if mt_row["crisis_f1"] >= lr_row["crisis_f1"] else "TF-IDF + LR"
topic_winner = "Multi-task" if mt_row["topic_f1"] >= b5_row["topic_f1"] else "B5 topics"

justification = f"""
**Paragraph 1 — Task performance:** On the human-reviewed eval set ({len(eval_gold)} posts), the multi-task model achieves crisis macro-F1={mt_row['crisis_f1']:.3f} vs {lr_row['crisis_f1']:.3f} for TF-IDF+LR crisis detection; topic macro-F1={mt_row['topic_f1']:.3f} vs {b5_row['topic_f1']:.3f} for B5 BERTopic (shared.topics); sentiment MAE={mt_row['sentiment_mae']:.3f} vs {b4_row['sentiment_mae']:.3f} for B4 LR sentiment. {crisis_winner} is stronger on crisis — the task with no existing basic-layer module.

**Paragraph 2 — Unified vs modular pipeline:** The multi-task model runs one encoder pass for all three tasks. The pipeline baseline stitches together three separate basic-layer components (B4 LR pkl, B5 BERTopic via fit_bertopic/assign_thread_topics, crisis LR trained on pseudo-labels). The unified model replaces three inference paths with one, which is operationally simpler for monitoring.

**Paragraph 3 — Architecture evidence + monitoring relevance:** A Q-Former-style frozen-encoder attempt was run first, but it showed ~42M trainable parameters (freezing failed) and collapsed crisis/sentiment heads on this small dataset, so we switched to partial fine-tuning. Crisis severity has no B-layer equivalent until this notebook — the LR baseline validates that a basic technique on pseudo-labels underperforms (or matches) the multi-task model, justifying a unified model for real-time scoring via predict(). Severity ≥2 triggers ReAct/CoT and RAG in Phase 2.
"""
print(justification)

## 7. Limitations

In [ ]:
from IPython.display import Markdown, display

limitations = (
    f"Only the top RoBERTa layers are fine-tuned on a small pseudo-labeled dataset "
    f"(train pool size = {len(train_pool)}), so sarcasm, memes, and community-specific slang may still "
    f"be misread. Training labels come from **ChatGPT pseudo-labeling** (external, one-time) and inherit "
    f"whatever bias or blind spots that model has. Severity-3 crises remain rare even after stratified "
    f"oversampling, so real-world recall on true crises is uncertain beyond what the {len(eval_gold)}-post "
    f"human-reviewed eval set can show; the abandoned Q-Former run underscores that data size, not model "
    f"capacity, is the current bottleneck."
)
display(Markdown(limitations))

## 8. Pipeline connection

Layer 2's cleaned Reddit text feeds this notebook (we use `load_sample()` and `clean_for_llm()` to mirror the preprocessing pipeline). The exported `predict()` function in `advanced/layer3_llm/predict.py` is called by Layer 4's crisis monitor (A5 ReAct) to score incoming posts in real time. Posts with `crisis_severity` above a configured threshold trigger ReAct/CoT/ToT reasoning and A2 RAG retrieval for evidence-backed crisis reports in Phase 2.

## 9. Export function

Phase 2 imports `predict()` directly — no re-training required.

In [ ]:
from advanced.layer3_llm.predict import predict

samples = [
    "OpenAI API has been down for 6 hours, our production app is completely broken.",
    "GPT-4o voice mode is incredible, best update in months!",
    "Subscription price keeps going up, not sure it's worth it anymore.",
]
for s in samples:
    print(json.dumps({"text": s[:80], **predict(s)}, indent=2))
    print()